In [ ]:
#run these commands if you're using google colab
'''
!pip install tensorflow-gpu==2.10.0
!pip install gym==0.26.0
!pip install tensorflow-probability==0.12.0
!pip install gym[atari,accept-rom-license]==0.26.2
'''


In [ ]:
import tensorflow as tf

tf.config.list_physical_devices("GPU")
import warnings
warnings.filterwarnings('ignore')
from contextlib import redirect_stdout, redirect_stderr
from tensorflow import keras
import sys
import contextlib
tf.__version__

In [ ]:
import gym
import numpy as np
import matplotlib.pyplot as plt

import gym
print(gym.__version__)
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from IPython import display

In [ ]:
import tensorflow_probability as tfp


In [ ]:
def preprocess(image):
    """Preprocess a frame from the game."""
    image = image[35:195]  # Crop
    image = image[::2, ::2, 0]  # Downsample by a factor of 2
    image[image == 144] = 0  # Erase background (background type 1)
    image[image == 109] = 0  # Erase background (background type 2)
    image[image != 0] = 1  # Set paddles and ball to 1, everything else to 0
    return np.reshape(image.astype(np.float32).ravel(), [80, 80])


# Neural Network

In [ ]:

# hyperparameters
learning_rate = 0.0001
gamma = 0.99

env = gym.make("Pong-v0")

In [ ]:
env.reset()[0].shape

In [ ]:

model_policy = tf.keras.Sequential()
model_policy.add(tf.keras.Input(shape=(80, 80, 2)))

model_policy.add(tf.keras.layers.Flatten())
model_policy.add(tf.keras.layers.Dense(256, activation='linear'))
#model.add(tf.keras.layers.Dense(256, activation='linear'))
model_policy.add(tf.keras.layers.Dense(2, activation='softmax'))
optimizer_policy = tf.keras.optimizers.Adam(learning_rate)


In [ ]:
'''
class Value(tf.keras.Model):
    def __init__(self):
        super(Value, self).__init__()
        self.l1 = keras.Input(shape=(80, 80, 2))
        self.l2 = keras.layers.Dense(12, activation='relu', input_shape=(32,))
        self.l3  = keras.layers.Dense(1, activation='linear')

    def forward(self, x):
        x = self.l1(x)
        x = self.l2(x)
        x = self.l3(x)

        return x

model_value  = Value()
'''
model_value = tf.keras.Sequential()
model_value.add(tf.keras.Input(shape=(80, 80, 2)))
model_value.add(tf.keras.layers.Flatten())
model_value.add(tf.keras.layers.Dense(32, activation='linear'))
model_value.add(tf.keras.layers.Dense(1, activation='linear'))

optimizer_value = tf.keras.optimizers.Adam(learning_rate)


## Load model weights from episode Run 4 - episode 500 previously.

In [ ]:
from tensorflow.keras.models import load_model

model_policy = load_model('policy_model_400')


## Testing the Model

In [ ]:


env = gym.make("Pong-v0")

episode_totalrewards = []


for episode in range(500):
    #reward_history_eps = []
    terminated = False
    truncated = False
    episode_reward = 0
    state = preprocess(env.reset()[0])

    prev_state = state

    while (not terminated) and (not truncated):
        # sampling an action from the policy
        with open('output.log', 'w') as log_file:
            with contextlib.redirect_stdout(log_file), contextlib.redirect_stderr(log_file):
                if episode_reward==0:
                    combined_states = tf.stack([tf.convert_to_tensor(prev_state), tf.convert_to_tensor(prev_state)], axis=-1)
                    action_prob = model_policy(np.expand_dims(combined_states, axis=0))[0]
                else:
                    action_prob = model_policy(np.expand_dims(combined_states, axis=0))[0]

        dist = tfp.distributions.Categorical(probs=action_prob, dtype=tf.float32)
        action = dist.sample()
        #print(int(action))
        if int(action)==0:
            a=2
        elif int(action)==1:
            a=3
        next_state, reward, terminated, truncated, info = env.step(a)
        next_state = preprocess(next_state)
        combined_states = tf.stack([tf.convert_to_tensor(prev_state), tf.convert_to_tensor(next_state)], axis=-1)
        prev_state = next_state
        #reward_history_eps.append(reward)
        episode_reward = episode_reward+ reward

    episode_totalrewards.append(episode_reward)
    #print(f"Episode: {episode}, Reward: {episode_reward}")


# Close the environment
env.close()


In [ ]:
mean_reward = np.mean(episode_totalrewards)
std_reward = np.std(episode_totalrewards)


# Create a histogram
plt.hist(episode_totalrewards, bins=10, color='orange', alpha=0.7)
plt.xlabel('Episode Total Rewards')
plt.ylabel('Number of Episodes')
plt.title('Histogram of Episode Rewards _based on run1 model')
plt.legend([f'Rewards (Mean: {mean_reward:.2f}, Std Dev: {std_reward:.2f})'])
plt.grid(True)




# Show the histogram
plt.show()
